In [1]:
import os

from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv("../.env")

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

#### Zero Shot Prompting Method Orignal Output

In [ ]:
# def get_completion(prompt: str, deployment_name: str = deployment_name) -> dict:
#     """Get a chat completion from Azure OpenAI.

#     Args:
#         prompt: User input prompt.
#         deployment_name: The deployment name you gave your model in Azure portal.

#     Returns:
#         Full response object as dict, or error dict on failure.
#     """
#     try:
#         messages = [{"role": "user", "content": prompt}]

#         response = client.chat.completions.create(
#             model=deployment_name,
#             messages=messages,
#             temperature=0.1,
#             top_p=0.8,
#             max_tokens=512, 
#         )

#         return response.model_dump()

#     except Exception as e:
#         return {"error": str(e)}

In [ ]:
# prompts: list[str] = [
#     "Imagine you are a detective trying to solve a mystery.",
#     "You arrive at the crime scene and start looking for clues.",
#     "You find a strange object at the crime scene. What is it?",
#     "How does this object relate to the crime?",
#     "Who do you think is the suspect and why?",
# ]

In [ ]:
# for prompt in prompts:
#     response = get_completion(prompt)
#     print(f"Prompt: {prompt}")
#     print(f"Response: {response}")
#     print()

Prompt: Imagine you are a detective trying to solve a mystery.
Response: {'id': 'chatcmpl-DWThf1s5MWMHxjwd78WdW32TGF3SN', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'Absolutely! I’m ready to put on my detective hat. What’s the mystery we’re trying to solve? Give me the details, and I’ll start piecing together the clues.', 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': None}, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'created': 1776633047, 'model': 'gpt-4.1-mini-2025-04-14', 'object': 'chat.completion', 'service_tier': 'default', 'system_fingerprint': 'fp_

In [5]:
#As seen above: Each call to get_completion(prompt) is stateless.
'''
So the model:
--Does NOT remember previous prompts
--Treats every line like a brand new conversation

That’s why:
It asks for context again
It forgets the “object”
It can’t identify the suspect

'''

'\nSo the model:\n--Does NOT remember previous prompts\n--Treats every line like a brand new conversation\n\nThat’s why:\nIt asks for context again\nIt forgets the “object”\nIt can’t identify the suspect\n\n'

#### Zero Shot Prompting Method

In [6]:
#First lets test and return only content
def get_completion(prompt, deployment_name=deployment_name):
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.choices[0].message.content  

    except Exception as e:
        return f"Error: {str(e)}"

In [7]:
for prompt in prompts:
    response = get_completion(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()

Prompt: Imagine you are a detective trying to solve a mystery.
Response: Absolutely! I’m ready to put on my detective hat. What’s the mystery we’re trying to solve?

Prompt: You arrive at the crime scene and start looking for clues.
Response: You arrive at the crime scene, a dimly lit room with signs of a struggle. The air is thick with tension. You begin your search for clues.

What do you want to examine first?

- The body
- The surrounding area
- Any objects nearby
- Ask witnesses or bystanders

Prompt: You find a strange object at the crime scene. What is it?
Response: Could you please provide more details or describe the object you found? That way, I can help identify it or suggest what it might be.

Prompt: How does this object relate to the crime?
Response: Could you please provide more details or specify which object you are referring to? That way, I can give you a more accurate explanation of how it relates to the crime.

Prompt: Who do you think is the suspect and why?
Respon

In [8]:
'''
1. First response → rich story
The model creates its own scenario:

Blackwood Manor
Missing manuscript
Defined suspects

Second prompt → completely new scene

“You arrive at the crime scene…”

Now suddenly:

New location: apartment
No manuscript
No suspects

> That means → memory is lost
Later prompts → confusion
Model says:
 “what object?”
 “missing context”

> Because:
It never saw previous outputs
We didn’t pass them back'''

'''
Summary of what happened:
first answer looks “smart” > Because our first prompt: “Imagine you are a detective…”

This is open-ended, so model: invents everything, drives the narrative

But later prompts:
depend on previous context
which is missing
'''

'\nSummary of what happened:\nfirst answer looks “smart” > Because our first prompt: “Imagine you are a detective…”\n\nThis is open-ended, so model: invents everything, drives the narrative\n\nBut later prompts:\ndepend on previous context\nwhich is missing\n'

#### Chain of thought prompting Method

In [9]:
#OPTION 3 — Maintain conversation history
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=512
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [10]:
messages = [
    {"role": "system", "content": "You are a detective solving a mystery step by step."}
]

prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")

Prompt: Start the investigation.
Response: Understood. To begin the investigation, I need some initial information:

1. What is the nature of the mystery or crime?
2. Where and when did it occur?
3. Are there any known suspects, witnesses, or evidence so far?
4. Any other relevant details you can provide.

Please provide these details so I can start piecing together the case.

Prompt: You arrive at the crime scene.
Response: I arrive at the crime scene. 

Step 1: Assess the scene and secure it to avoid contamination of evidence.  
Step 2: Observe and document the environment—note the location, time, and any immediate visible clues.  
Step 3: Identify and speak to any witnesses or individuals present.  
Step 4: Begin collecting physical evidence—photographs, fingerprints, any objects out of place, etc.

Please describe the scene or provide details about what I see upon arrival.

Prompt: You find a strange object. Describe it.
Response: At the crime scene, I notice a strange object lying

In [11]:
'''
Our setup produced:

--Sequential reasoning across steps
--Evidence → inference → conclusion flow
--Consistent story memory
--Logical suspect identification

Output:
Pen found
Initials identified
Guest log matched
Suspect inferred

>> That is implicit chain-of-thought reasoning

What We did NOT strictly achieve > explicit CoT format, like:

“Step 1: …”
“Reasoning: …”
“Therefore: …”

Instead, the model:
--Embedded reasoning naturally inside narration
--Mixed story + logic
This is called:
>> Narrative reasoning, not strict CoT prompting
'''

'\nOur setup produced:\n\n--Sequential reasoning across steps\n--Evidence → inference → conclusion flow\n--Consistent story memory\n--Logical suspect identification\n\nOutput:\nPen found\nInitials identified\nGuest log matched\nSuspect inferred\n\n>> That is implicit chain-of-thought reasoning\n\nWhat We did NOT strictly achieve > explicit CoT format, like:\n\n“Step 1: …”\n“Reasoning: …”\n“Therefore: …”\n\nInstead, the model:\n--Embedded reasoning naturally inside narration\n--Mixed story + logic\nThis is called:\n>> Narrative reasoning, not strict CoT prompting\n'

In [12]:
#Improved System prompt (strict COT)
messages = [
    {
        "role": "system",
        "content": """You are a detective AI.

For every step:
- Do NOT ask the user questions
- Continue the story automatically
- Show reasoning explicitly in this format:

Observation:
Reasoning:
Conclusion:

Keep the story consistent across steps."""
    }
]

In [13]:
prompts = [
    "Start the investigation.",
    "You arrive at the crime scene.",
    "You find a strange object. Describe it.",
    "How does it relate to the crime?",
    "Who is the suspect?"
]

for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = get_completion(messages)
    
    messages.append({"role": "assistant", "content": response})
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
#Evidence → reasoning → conclusion (consistently across turns)

Prompt: Start the investigation.
Response:
Observation: The investigation has just begun, with no initial information or context provided.
Reasoning: In order to proceed effectively, I need to establish the setting, identify the crime or mystery, and gather initial evidence or reports.
Conclusion: I will start by defining the nature of the case and collecting preliminary details from the scene or witnesses to set the investigation in motion.

---

The detective arrives at the scene of the incident—a dimly lit alley behind a downtown bar. The victim, a middle-aged man, lies motionless on the ground. Nearby, there is a broken bottle and faint traces of blood. No witnesses are visible at the moment, but the detective notices a security camera mounted on a nearby building.

The detective begins by securing the scene and reviewing the security footage to gather clues on what transpired.

Prompt: You arrive at the crime scene.
Response:
Observation: The detective has physically arrived at th

In [14]:
#Paramerizing
def get_completion(messages, deployment_name=deployment_name):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            temperature=0.7,
            max_tokens=800
        )

        choice = response.choices[0]

        # Normal case
        if choice.message and choice.message.content:
            return choice.message.content.strip()

        # Edge case (no content)
        return f"[No content | finish_reason={choice.finish_reason}]"

    except Exception as e:
        return f"Error: {str(e)}"

def safe_get_completion(messages):
    for _ in range(3):
        res = get_completion(messages)

        if res and \
           not res.startswith("[") and \
           not res.startswith("Error") and \
           not res.startswith("Fallback") and \
           len(res.strip()) > 20:
            return res

    return "Fallback: Model failed to generate meaningful response."

In [ ]:
# crime_type = input("Enter crime type: ")
crime_type = 'Fraud'
# location = input("Enter location: ")#######2333221a
location = 'China'
# object_hint = input("Enter a clue/object (optional): ")
object_hint = 'Money'
object_hint = object_hint if object_hint else "a suspicious object"

In [16]:
messages = [
    {
        "role": "system",
        "content": f"""
You are a detective AI solving a {crime_type} case at {location}.
You MUST ALWAYS respond with meaningful content.
If uncertain, invent reasonable details.

Format:
Observation:
Reasoning:
Conclusion:

Never return empty output.
Never skip a response.
"""
    }
]

In [17]:
prompts = [
    f"Start the investigation of a {crime_type} at {location}. Describe the victim, surroundings, and key initial clues.",

    f"You arrive at the crime scene in {location}. Describe what you observe in detail including the body, environment, and any immediate evidence.",

    f"Describe a strange object such as {object_hint} found at the scene.",

    "Explain how this object relates to the crime.",

    "Identify the most likely suspect based on all evidence."
]

In [18]:
#With memory handling
for prompt in prompts:
    messages.append({"role": "user", "content": prompt})
    
    response = safe_get_completion(messages)
    
    print(f"Prompt: {prompt}")
    print(f"Response:\n{response}\n")
    
    # Only store valid responses
    if not response.startswith("Fallback") and not response.startswith("["):
        messages.append({"role": "assistant", "content": response})
    else:
        print("Skipping bad response, not adding to history")

Prompt: Start the investigation of a Fraud at China. Describe the victim, surroundings, and key initial clues.
Response:
Observation:
The victim is Mr. Li Wei, a mid-level executive at a well-known tech company based in Shenzhen, China. He reported unauthorized transactions amounting to approximately 2 million RMB from his corporate accounts over the past two months. Mr. Li lives in a high-rise apartment complex in the Futian district, a bustling commercial area. His office is located in the company’s headquarters, a modern glass building equipped with advanced security systems. Initial interviews reveal that Mr. Li recently received several suspicious emails requesting confidential information. Additionally, access logs show unusual login times and IP addresses from outside China, which do not correlate with Mr. Li’s normal working hours or travel records.

Reasoning:
The victim is a corporate executive with access to significant financial resources, making him a prime target for frau

#### Another example

In [2]:
prompt = """
let's analyze the sentiment of the review step by step

1. Identify the Positive aspect of the review and give a score from 10
2. Identify the Negative aspect of the review and give a score from 10
3. Weight the positive and negative aspect to determine the overall sentiment.
4. provide the final sentiment classification with justification for scores used in above steps.

Review: "The product is very well designed product"
"""

In [3]:
response = get_completion(prompt)

NameError: name 'get_completion' is not defined